<a href="https://colab.research.google.com/github/shreyanayak440/portfolio/blob/main/shreya.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
Multimodal Cyber-Threat Detector
================================
An end-to-end PyTorch training script that fuses
1) **voice/audio** signals (Wav2Vec2 embeddings),
2) optional **free-text** context (DistilBERT embeddings), and
3) **tabular/numeric** telemetry (standardized),
into a single classifier for binary cyber‑threat detection.

Dataset format (CSV)
--------------------
Provide a CSV (e.g., data/train.csv) with columns like:

    audio_path,context_text,num_packets,num_failed_logins,num_bytes,label
    path/to/clip1.wav,"Suspicious login from new device",120,5,56000,1
    path/to/clip2.wav,"Routine backup operation",30,0,120000,0

- `audio_path` : path to a mono/stereo audio file (wav/mp3/flac, etc.).
- `context_text` (optional): any textual metadata/log summary (can be empty).
- Numeric features must be prefixed with `num_` (e.g., `num_packets`).
- `label` : 0 (benign) or 1 (threat).

Install requirementsq
--------------------
    pip install torch torchvision torchaudio librosa transformers pandas scikit-learn tqdm

Notes
-----
- The script downloads small pretrained models from Hugging Face on first run:
  - Audio: `facebook/wav2vec2-base`
  - Text : `distilbert-base-uncased`
- By default we **freeze** the backbone encoders and only train the fusion head (fast + stable).
- Works even if a modality is missing (e.g., no `context_text` or no numeric columns).

Usage
-----
    python multimodal_cyber_threat.py \
        --train_csv data/train.csv \
        --val_csv data/val.csv \
        --epochs 10 --batch_size 8 --lr 3e-4 \
        --audio_max_sec 8 --save_path runs/model.pt

    # Inference on a CSV (same columns, label ignored if present)
    python multimodal_cyber_threat.py --predict_csv data/test.csv --load_path runs/model.pt

"""
import os
import argparse
import warnings
from typing import List, Dict, Optional, Tuple

import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import librosa
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score

from transformers import (
    AutoFeatureExtractor,
    Wav2Vec2Model,
    AutoTokenizer,
    AutoModel,
)

warnings.filterwarnings("ignore")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ------------------------------
# Utilities
# ------------------------------

def read_audio(path: str, target_sr: int = 16000, max_sec: Optional[int] = None) -> np.ndarray:
    """Load audio as mono float32 at target_sr; optionally cap duration."""
    if not os.path.exists(path):
        raise FileNotFoundError(f"Audio file not found: {path}")
    y, sr = librosa.load(path, sr=target_sr, mono=True)
    if max_sec is not None:
        max_len = int(target_sr * max_sec)
        if len(y) > max_len:
            y = y[:max_len]
    return y.astype(np.float32)


def detect_numeric_columns(df: pd.DataFrame) -> List[str]:
    # pick columns starting with 'num_' and numeric dtype
    cols = [c for c in df.columns if c.startswith("num_")]
    return [c for c in cols if pd.api.types.is_numeric_dtype(df[c])]


# ------------------------------
# Dataset
# ------------------------------
class ThreatDataset(Dataset):
    def __init__(
        self,
        csv_path: str,
        feature_extractor: AutoFeatureExtractor,
        tokenizer: Optional[AutoTokenizer],
        numeric_cols: Optional[List[str]] = None,
        audio_max_sec: Optional[int] = None,
        is_inference: bool = False,
    ):
        self.df = pd.read_csv(csv_path)
        if "audio_path" not in self.df.columns:
            raise ValueError("CSV must include 'audio_path' column")
        self.has_text = "context_text" in self.df.columns and tokenizer is not None
        self.has_label = ("label" in self.df.columns) and (not is_inference)
        self.numeric_cols = numeric_cols if numeric_cols is not None else detect_numeric_columns(self.df)
        self.feature_extractor = feature_extractor
        self.tokenizer = tokenizer
        self.audio_max_sec = audio_max_sec

        # Prepare numeric scaler on this dataset (fit on train; for val/predict, pass a fitted scaler)
        self.scaler = None

    def set_scaler(self, scaler: Optional[StandardScaler]):
        self.scaler = scaler

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx: int) -> Dict:
        row = self.df.iloc[idx]

        # Audio
        audio = read_audio(row["audio_path"], target_sr=16000, max_sec=self.audio_max_sec)
        audio_inputs = self.feature_extractor(audio, sampling_rate=16000, return_tensors="pt")
        audio_tensor = audio_inputs["input_values"][0]  # (T,)

        # Text (optional)
        text_inputs = None
        if self.has_text:
            text = str(row.get("context_text", ""))
            text_inputs = self.tokenizer(
                text,
                truncation=True,
                max_length=128,
                padding="max_length",
                return_tensors="pt",
            )

        # Numeric (optional)
        numeric = None
        if self.numeric_cols:
            numeric_vals = row[self.numeric_cols].astype(float).values.astype(np.float32)
            if self.scaler is not None:
                numeric_vals = self.scaler.transform([numeric_vals])[0]
            numeric = torch.tensor(numeric_vals, dtype=torch.float32)

        label = None
        if self.has_label:
            label = torch.tensor(int(row["label"]), dtype=torch.long)

        return {
            "audio": audio_tensor,
            "text_inputs": text_inputs,
            "numeric": numeric,
            "label": label,
        }


def collate_fn(batch: List[Dict]) -> Dict:
    # Pad audio to longest length in batch
    audios = [b["audio"] for b in batch]
    max_len = max(a.shape[-1] for a in audios)
    padded = []
    attn_mask = []
    for a in audios:
        pad_len = max_len - a.shape[-1]
        if pad_len > 0:
            a = torch.nn.functional.pad(a, (0, pad_len))
        padded.append(a)
        attn_mask.append(torch.ones_like(a))
    audio_batch = torch.stack(padded, dim=0)
    audio_attn = torch.stack(attn_mask, dim=0)

    # Text
    if batch[0]["text_inputs"] is not None:
        input_ids = torch.stack([b["text_inputs"]["input_ids"][0] for b in batch], dim=0)
        attention_mask = torch.stack([b["text_inputs"]["attention_mask"][0] for b in batch], dim=0)
        text_batch = {"input_ids": input_ids, "attention_mask": attention_mask}
    else:
        text_batch = None

    # Numeric
    if batch[0]["numeric"] is not None:
        numeric_batch = torch.stack([b["numeric"] for b in batch], dim=0)
    else:
        numeric_batch = None

    # Labels
    labels = None
    if batch[0]["label"] is not None:
        labels = torch.stack([b["label"] for b in batch], dim=0)

    return {
        "audio": audio_batch,
        "audio_attn": audio_attn,
        "text": text_batch,
        "numeric": numeric_batch,
        "labels": labels,
    }


# ------------------------------
# Model
# ------------------------------
class FusionNet(nn.Module):
    def __init__(
        self,
        audio_model_name: str = "facebook/wav2vec2-base",
        text_model_name: Optional[str] = "distilbert-base-uncased",
        numeric_dim: int = 0,
        freeze_backbones: bool = True,
        dropout: float = 0.2,
    ):
        super().__init__()
        # Audio encoder
        self.audio_model = Wav2Vec2Model.from_pretrained(audio_model_name)
        audio_dim = self.audio_model.config.hidden_size
        if freeze_backbones:
            for p in self.audio_model.parameters():
                p.requires_grad = False

        # Text encoder (optional)
        self.has_text = text_model_name is not None
        text_dim = 0
        if self.has_text:
            self.text_model = AutoModel.from_pretrained(text_model_name)
            text_dim = self.text_model.config.hidden_size
            if freeze_backbones:
                for p in self.text_model.parameters():
                    p.requires_grad = False

        self.numeric_dim = numeric_dim

        fusion_in = audio_dim + text_dim + numeric_dim
        self.classifier = nn.Sequential(
            nn.Linear(fusion_in, 512),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 2),
        )

    def forward(self, audio, audio_attn, text_batch=None, numeric_batch=None):
        # Audio: mean-pool last hidden state
        a_hidden = self.audio_model(audio, attention_mask=audio_attn).last_hidden_state  # B x T x D
        a_vec = a_hidden.mean(dim=1)  # B x D

        feats = [a_vec]

        # Text: CLS token from transformer
        if self.has_text and text_batch is not None:
            t_out = self.text_model(**text_batch)
            if hasattr(t_out, "last_hidden_state"):
                t_cls = t_out.last_hidden_state[:, 0, :]  # B x D
            else:  # some models use pooler_output
                t_cls = t_out.pooler_output
            feats.append(t_cls)

        # Numeric
        if numeric_batch is not None:
            feats.append(numeric_batch)

        fused = torch.cat(feats, dim=1)
        logits = self.classifier(fused)
        return logits


# ------------------------------
# Training & Evaluation
# ------------------------------

def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    losses = []
    for batch in tqdm(loader, desc="train", leave=False):
        audio = batch["audio"].to(DEVICE)
        audio_attn = batch["audio_attn"].to(DEVICE)
        text = None
        if batch["text"] is not None:
            text = {k: v.to(DEVICE) for k, v in batch["text"].items()}
        numeric = batch["numeric"].to(DEVICE) if batch["numeric"] is not None else None
        labels = batch["labels"].to(DEVICE)

        optimizer.zero_grad()
        logits = model(audio, audio_attn, text, numeric)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    return float(np.mean(losses))


def evaluate(model, loader):
    model.eval()
    all_y, all_p, all_pred = [], [], []
    with torch.no_grad():
        for batch in tqdm(loader, desc="eval", leave=False):
            audio = batch["audio"].to(DEVICE)
            audio_attn = batch["audio_attn"].to(DEVICE)
            text = None
            if batch["text"] is not None:
                text = {k: v.to(DEVICE) for k, v in batch["text"].items()}
            numeric = batch["numeric"].to(DEVICE) if batch["numeric"] is not None else None
            labels = batch["labels"].to(DEVICE) if batch["labels"] is not None else None

            logits = model(audio, audio_attn, text, numeric)
            probs = F.softmax(logits, dim=1)[:, 1]
            preds = (probs > 0.5).long()

            if labels is not None:
                all_y.append(labels.cpu().numpy())
            all_p.append(probs.cpu().numpy())
            all_pred.append(preds.cpu().numpy())

    if len(all_y) > 0:
        y = np.concatenate(all_y)
        p = np.concatenate(all_p)
        pred = np.concatenate(all_pred)
        try:
            auc = roc_auc_score(y, p)
        except ValueError:
            auc = float("nan")
        acc = accuracy_score(y, pred)
        f1 = f1_score(y, pred)
        return {"auc": auc, "acc": acc, "f1": f1}
    else:
        return {}


def predict_to_csv(model, loader, out_path: str):
    model.eval()
    rows = []
    with torch.no_grad():
        for batch in tqdm(loader, desc="predict", leave=False):
            audio = batch["audio"].to(DEVICE)
            audio_attn = batch["audio_attn"].to(DEVICE)
            text = None
            if batch["text"] is not None:
                text = {k: v.to(DEVICE) for k, v in batch["text"].items()}
            numeric = batch["numeric"].to(DEVICE) if batch["numeric"] is not None else None

            logits = model(audio, audio_attn, text, numeric)
            probs = F.softmax(logits, dim=1)[:, 1].cpu().numpy()
            rows.extend(probs.tolist())

    pd.DataFrame({"threat_prob": rows}).to_csv(out_path, index=False)


# ------------------------------
# Main
# ------------------------------

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--train_csv", type=str, default=None)
    parser.add_argument("--val_csv", type=str, default=None)
    parser.add_argument("--predict_csv", type=str, default=None)

    parser.add_argument("--audio_model", type=str, default="facebook/wav2vec2-base")
    parser.add_argument("--text_model", type=str, default="distilbert-base-uncased")
    parser.add_argument("--no_text", action="store_true", help="Disable text modality even if provided")

    parser.add_argument("--epochs", type=int, default=10)
    parser.add_argument("--batch_size", type=int, default=8)
    parser.add_argument("--lr", type=float, default=3e-4)
    parser.add_argument("--audio_max_sec", type=int, default=8)
    parser.add_argument("--freeze_backbones", action="store_true")
    parser.add_argument("--no_freeze_backbones", dest="freeze_backbones", action="store_false")
    parser.set_defaults(freeze_backbones=True)

    parser.add_argument("--save_path", type=str, default="runs/model.pt")
    parser.add_argument("--load_path", type=str, default=None)
    parser.add_argument("--pred_out", type=str, default="runs/predictions.csv")

    args = parser.parse_args()

    # Which modalities?
    use_text = (not args.no_text) and (args.text_model is not None)

    # Feature extractors / tokenizers
    feat_extractor = AutoFeatureExtractor.from_pretrained(args.audio_model)
    tokenizer = AutoTokenizer.from_pretrained(args.text_model) if use_text else None

    # Training / Validation
    if args.train_csv is not None and args.val_csv is not None:
        train_df = pd.read_csv(args.train_csv)
        val_df = pd.read_csv(args.val_csv)
        numeric_cols = detect_numeric_columns(train_df)

        train_ds = ThreatDataset(
            args.train_csv, feat_extractor, tokenizer, numeric_cols, args.audio_max_sec, is_inference=False
        )
        val_ds = ThreatDataset(
            args.val_csv, feat_extractor, tokenizer, numeric_cols, args.audio_max_sec, is_inference=False
        )

        # Fit scaler on train numeric columns
        scaler = None
        if numeric_cols:
            scaler = StandardScaler()
            scaler.fit(train_df[numeric_cols].astype(float).values)
        train_ds.set_scaler(scaler)
        val_ds.set_scaler(scaler)

        train_loader = DataLoader(train_ds, batch_size=args.batch_size, shuffle=True, collate_fn=collate_fn)
        val_loader = DataLoader(val_ds, batch_size=args.batch_size, shuffle=False, collate_fn=collate_fn)

        model = FusionNet(
            audio_model_name=args.audio_model,
            text_model_name=(args.text_model if use_text else None),
            numeric_dim=len(numeric_cols),
            freeze_backbones=args.freeze_backbones,
        ).to(DEVICE)

        optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=args.lr)
        criterion = nn.CrossEntropyLoss()

        best_auc = -1
        os.makedirs(os.path.dirname(args.save_path), exist_ok=True)

        for epoch in range(1, args.epochs + 1):
            tr_loss = train_one_epoch(model, train_loader, optimizer, criterion)
            metrics = evaluate(model, val_loader)
            auc = metrics.get("auc", float("nan"))
            acc = metrics.get("acc", float("nan"))
            f1 = metrics.get("f1", float("nan"))
            print(f"Epoch {epoch}: train_loss={tr_loss:.4f} | val_auc={auc:.4f} acc={acc:.4f} f1={f1:.4f}")

            if not np.isnan(auc) and auc > best_auc:
                best_auc = auc
                torch.save({
                    "model_state": model.state_dict(),
                    "audio_model": args.audio_model,
                    "text_model": (args.text_model if use_text else None),
                    "numeric_dim": len(numeric_cols),
                    "freeze_backbones": args.freeze_backbones,
                    "scaler_mean": (train_ds.scaler.mean_.tolist() if train_ds.scaler is not None else None),
                    "scaler_scale": (train_ds.scaler.scale_.tolist() if train_ds.scaler is not None else None),
                    "numeric_cols": numeric_cols,
                }, args.save_path)
                print(f"Saved best model to {args.save_path}")

    # Prediction-only mode
    if args.predict_csv is not None:
        if args.load_path is None or not os.path.exists(args.load_path):
            raise ValueError("--predict_csv requires a valid --load_path to a trained checkpoint")
        ckpt = torch.load(args.load_path, map_location="cpu")

        feat_extractor = AutoFeatureExtractor.from_pretrained(ckpt.get("audio_model", "facebook/wav2vec2-base"))
        text_model_name = ckpt.get("text_model", None)
        tokenizer = AutoTokenizer.from_pretrained(text_model_name) if text_model_name is not None else None

        numeric_cols = ckpt.get("numeric_cols", [])
        ds = ThreatDataset(
            args.predict_csv, feat_extractor, tokenizer, numeric_cols, args.audio_max_sec, is_inference=True
        )
        # Restore scaler
        mean, scale = ckpt.get("scaler_mean", None), ckpt.get("scaler_scale", None)
        if mean is not None and scale is not None and numeric_cols:
            scaler = StandardScaler()
            scaler.mean_ = np.array(mean)
            scaler.scale_ = np.array(scale)
            ds.set_scaler(scaler)

        loader = DataLoader(ds, batch_size=args.batch_size, shuffle=False, collate_fn=collate_fn)

        model = FusionNet(
            audio_model_name=ckpt.get("audio_model", "facebook/wav2vec2-base"),
            text_model_name=text_model_name,
            numeric_dim=len(numeric_cols),
            freeze_backbones=ckpt.get("freeze_backbones", True),
        ).to(DEVICE)
        model.load_state_dict(ckpt["model_state"], strict=True)

        os.makedirs(os.path.dirname(args.pred_out), exist_ok=True)
        predict_to_csv(model, loader, args.pred_out)
        print(f"Predictions saved to {args.pred_out}")
